# Project 2:  Markov Chain Text Generation

In [ ]:
"""
project02.ipynb

Implements a Markov Chain algorithm to generate new text in the style of
Shakespeare's sonnets. build_markov_model function reads in text and constructs a
model of word transition probabilities. For a given state (the previous
word, or the previous `order` words for higher-order models), it tracks
how often each possible next word follows that state, using *S* and *E*
markers to represent the start and end of a text. get_next_word function uses these
transition probabilities to randomly select a plausible next word from a
given state, weighted by how often that word actually followed the state
in the source text. generate_random_text function repeatedly calls get_next_word,
starting from the start state and continuing until the end state is
generated, to produce a full block of newly generated text.

For the "Pick Your Poison" section, the model is built from Shakespeare's
sonnets (data/sonnets.txt), read in and split into individual sonnets using
blank lines as separators, with each sonnet fed into build_markov_model
using an order of 2. The resulting model is then used to generate new,
sonnet-style text.

Authors are Ahmed Salman, Selin Uenal, and Ildiko Polyak.
"""

In [34]:
import numpy as np
from pprint import pprint

## build_markov_model (1st order, generalized to Nth order)

In [35]:
def build_markov_model(markov_model, text, order=1):
    """
    Args:
        markov_model (dict of dicts): existing Markov model
        text (str): text to add to the Markov model
        order (int): number of previous words used as the state

    Returns:
        markov_model (dict of dicts): updated model
    """

    if markov_model is None:
        markov_model = {}

    # Split the current sonnet into words
    words = text.split()

    # add start and end states to words list
    words_with_states = (['*S*'] * order) + words + ['*E*']

    # Loop through text and update markov model 
    for position in range(len(words_with_states) - order):

        # store 1st order state as a string
        if order == 1:
            current_state = words_with_states[position]
        # for order > 1, store state as a tuple
        else:
            current_state = tuple(words_with_states[position:position + order])

        next_word = words_with_states[position + order]

        if current_state not in markov_model:
            markov_model[current_state] = {}

        if next_word not in markov_model[current_state]:
            markov_model[current_state][next_word] = 0

        markov_model[current_state][next_word] += 1

    return markov_model


In [36]:
# Test: check build_markov_model against the "one fish two fish" example
test_model = build_markov_model({}, "one fish two fish red fish blue fish", order=1)
pprint(test_model)
#Confirmed:  returns {'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}

{'*S*': {'one': 1},
 'blue': {'fish': 1},
 'fish': {'*E*': 1, 'blue': 1, 'red': 1, 'two': 1},
 'one': {'fish': 1},
 'red': {'fish': 1},
 'two': {'fish': 1}}


## get_next_word

In [37]:
def get_next_word(current_state, markov_model, seed=42):
    """
    Args:
        current_state: the current word (or tuple of words) to look up
        markov_model (dict of dicts): the model
        seed (int): random seed for reproducibility

    Returns:
        next_word (str): randomly selected next word
    """
    # Get the possible next words and their counts
    next_word_counts = markov_model[current_state]

    # Add up all the counts for this current word to get a total
    total = sum(next_word_counts.values())

    next_word_probabilities = {}

    # Calculate the probability of each next word
    for word in next_word_counts:
        probability = next_word_counts[word] / total
        next_word_probabilities[word] = probability

    # Only set seed when one is provided
    if seed is not None:
        np.random.seed(seed)    

    # Randomly select the next word using the probabilities
    next_word = np.random.choice(list(next_word_probabilities.keys()), p=list(next_word_probabilities.values()))

    return next_word  #Return the selected word.


In [ ]:
#Test: build a tiny dictionary to check that get_next_word function works
tiny_test_model = {"fish": {"two": 1, "red": 1, "blue": 1, "*E*": 1}}
print(get_next_word("fish", tiny_test_model, seed=42))
#Confirmed:  returns "red" with seed=42

red


## generate_random_text

In [39]:
def generate_random_text(markov_model, seed=42):
    """
    Args:
        markov_model (dict of dicts): the model
        seed (int): random seed for reproducibility

    Returns:
        sentence (str): generated text
    """
    # Set the seed once for the full generated sentence
    np.random.seed(seed)

    # Initialize the current state and generated sentence
    current_state = next(iter(markov_model))
    generated_sentence = ""

    # Continue until the end state is selected
    while True:
        next_word = get_next_word(current_state, markov_model, seed=None)

        if next_word == "*E*":
            break

        generated_sentence = generated_sentence + next_word + " "

        if isinstance(current_state, tuple):
            current_state = current_state[1:] + (next_word,)
        else:
            current_state = next_word

    return generated_sentence

## Pick Your Poison: Sonnets

In [53]:
poison_markov_model = dict()
with open("data/sonnets.txt", "r") as poison_text:
    # Process the lines. Consider that sonnets are separated by an empty line. and create corpus.
    corpus = []
    sonnet = ''
    for line in poison_text:
        if line != '\n':
            sonnet = sonnet + line
        else:
            corpus.append(sonnet)
            sonnet = ''
    if sonnet:
        corpus.append(sonnet)
    for sonnet in corpus:
        poison_markov_model = build_markov_model(markov_model = poison_markov_model, text = sonnet, order=2)
print(generate_random_text(poison_markov_model, seed=7))

When most I wink, then do mine eyes best see, For all the treasure of his spring; For such a counterpart shall fame his wit, Making his style admired every where. Give my love that still, And you in Grecian tires are painted new: Speak of the east, Nor that full star that ushers in the world is grown so bad, Mad slanderers by mad ears believed be. That I have no end: Mine appetite I never saw that you were when first I hallow'd thy fair flower add the rank smell of weeds: But why of two oaths' breach do I find, Happy to have what thou dost review The very part was consecrate to thee: 'Thou single wilt prove none.' 


---
## Expected output (seed = 7)

When most I wink, then do mine eyes best see, For all the treasure of his spring; For such a counterpart shall fame his wit, Making his style admired every where. Give my love that still, And you in Grecian tires are painted new: Speak of the east, Nor that full star that ushers in the world is grown so bad, Mad slanderers by mad ears believed be. That I have no end: Mine appetite I never saw that you were when first I hallow'd thy fair flower add the rank smell of weeds: But why of two oaths' breach do I find, Happy to have what thou dost review The very part was consecrate to thee: 'Thou single wilt prove none.' 